# 🛢️ Segmentación de Pozos mediante Aprendizaje No Supervisado
## Módulo 4 · Clustering K-Means & Diagnósticos Visuales · Capacitación SLB

### Objetivos de la sesión:
1. Comprender la necesidad física y matemática de estandarizar variables de entrada.
2. Determinar la cantidad óptima de grupos (K) mediante el método del codo y **Yellowbrick**.
3. Entrenar e interpretar un modelo K-Means sobre 50 pozos petroleros.

## ¿Qué librerías cargaremos para realizar el análisis de clusters?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

sns.set_theme(style="whitegrid")

# 1. Carga y Descarga de Datos

## ¿Cómo descargamos el archivo `pozos_clustering.csv` directamente desde GitHub?

In [ ]:
!wget -q https://raw.githubusercontent.com/DavidPonce84/machine-learning-course/main/modulo_4_no_supervisado/data/pozos_clustering.csv -O pozos_clustering.csv

df_pozos = pd.read_csv('pozos_clustering.csv')
df_pozos.head()

## ¿Cómo inspeccionamos las diferencias de escala física entre caudales, presiones y KPIs?

In [ ]:
# TU CÓDIGO AQUÍ: Muestra un describe() de las columnas numéricas para ver las escalas físicas
df_pozos.describe().round(2)

## ¿Cómo exploramos visualmente la estructura y dispersión 2D de los pozos antes de agrupar?

> **💡 Importante para Clustering:** Visualizar la forma geométrica de los datos en el espacio de características nos permite identificar si existen agrupaciones naturales, formas esféricas u observables de densidad (clave para comparar K-Means vs DBSCAN).

In [ ]:
# Exploración visual de la relación Qo vs Water_Cut
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_pozos, x="Qo_bpd", y="Water_Cut", s=80, color="navy", alpha=0.7)
plt.title("Dispersión 2D: Caudal de Crudo (Qo) vs Corte de Agua (Water Cut)")
plt.xlabel("Qo_bpd (Barriles de Petróleo por Día)")
plt.ylabel("Water_Cut (Fracción 0-1)")
plt.show()

## ¿Cómo evaluamos las relaciones multivariadas entre todas las variables operativas con ?


In [ ]:
features = ['Qo_bpd', 'Qw_bpd', 'WHP_psi', 'BHP_psi', 'Water_Cut']

# Pairplot multivariado de las variables de producción
sns.pairplot(df_pozos[features], corner=True, diag_kind="kde")
plt.suptitle("Matriz de Dispersión Multivariada de Pozos", y=1.02)
plt.show()

> **🔍 Observación Correcta según la Tabla de Medias:**
> - **Cluster 0**: Pozos maduros con alta producción de agua y conificación (Water Cut = 0.83, Qw = 1,870.95 bpd).
> - **Cluster 1**: Pozos de alto rendimiento y bajo corte de agua (Qo = 2,084.64 bpd, Water Cut = 0.14, BHP = 2,143.19 psi).
> - **Cluster 2**: Pozos declinados de baja presión y bajo caudal (Qo = 248.20 bpd, BHP = 1,305.83 psi - Candidatos a BES/Gas Lift).

## ¿Cómo estandarizamos las variables de producción con `StandardScaler` (media=0, std=1)?

In [ ]:
features = ['Qo_bpd', 'Qw_bpd', 'WHP_psi', 'BHP_psi', 'Water_Cut']

# TU CÓDIGO AQUÍ: Instancia StandardScaler y transforma las variables en 'features'
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pozos[features])

df_scaled = pd.DataFrame(X_scaled, columns=features)
df_scaled.describe().round(3)

# 2. Selección de K Óptimo con Yellowbrick

## ¿Cómo instalamos la librería de diagnóstico visual de clusters `yellowbrick`?

In [ ]:
!pip install -q yellowbrick

## ¿Cómo determina el visualizador `KElbow` el punto exacto de la inercia (codo)?

In [ ]:
from yellowbrick.cluster import KElbowVisualizer

# TU CÓDIGO AQUÍ: Instancia KElbowVisualizer usando KMeans(random_state=42) y k=(2, 6)
model = KMeans(random_state=42)
visualizer = KElbowVisualizer(model, k=(2, 6))
visualizer.fit(X_scaled)
visualizer.show()

# 3. Entrenamiento y Caracterización

## ¿Cómo entrenamos el modelo definitivo de K-Means con K=3 y asignamos los clusters a cada pozo?

In [ ]:
# TU CÓDIGO AQUÍ: Entrena KMeans con n_clusters=3 y asigna las etiquetas al DataFrame df_pozos['Cluster']
kmeans = KMeans(n_clusters=3, random_state=42)
df_pozos['Cluster'] = kmeans.fit_predict(X_scaled)

df_pozos.groupby('Cluster').size()

## ¿Cómo caracterizamos el perfil físico y productivo promedio de cada grupo de pozos?

In [ ]:
# TU CÓDIGO AQUÍ: Agrupa df_pozos por 'Cluster' y calcula el promedio de cada variable operativa
df_pozos.groupby('Cluster')[features].mean().round(2)

> **🔍 Observación:**
> - **Cluster 0**: Pozos de alto rendimiento y bajo water-cut.
> - **Cluster 1**: Pozos maduros con alta producción de agua (conificación de agua).
> - **Cluster 2**: Pozos de baja presión y bajo caudal (candidatos a levantamiento artificial ESP/Gas Lift).